In [1]:
# ============================================================
# CELL 1 — SETUP
# Mount Drive and create Llama/RAG output folder
# ============================================================

!pip install -q "transformers>=4.48,<5" accelerate bitsandbytes \
    huggingface_hub sacrebleu rapidfuzz bert-score==0.3.13

from google.colab import drive
drive.mount("/content/drive")

import os, json, re, gc, unicodedata
import pandas as pd
import numpy as np
import torch
from pathlib import Path

# Find existing project folder
possible = [
    Path("/content/drive/MyDrive/Govt_Chatbot"),
    Path("/content/drive/MyDrive/Govt_Chatbots")
]

BASE = None

for p in possible:
    if (p / "RAG" / "test_questions.csv").exists():
        BASE = p
        break

if BASE is None:
    raise FileNotFoundError("Govt_Chatbot/RAG folder not found.")

RAG_DIR = BASE / "RAG"
OUT = BASE / "Llama" / "RAG"

OUT.mkdir(parents=True, exist_ok=True)

print("RAG folder:", RAG_DIR)
print("Output folder:", OUT)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 93.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 12.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
# ============================================================
# CELL 2 — LOAD TEST QUESTIONS + RETRIEVALS
# Uses already stored RAG retrievals
# ============================================================

TEST_PATH = RAG_DIR / "test_questions.csv"
RETRIEVAL_PATH = RAG_DIR / "retrievals.json"

tests = pd.read_csv(TEST_PATH).fillna("")

with open(RETRIEVAL_PATH, encoding="utf-8") as f:
    retrievals = json.load(f)

assert len(tests) == len(retrievals), \
    "Test questions and retrievals do not match."

assert "question" in tests.columns
assert "gold" in tests.columns

print("Test questions:", len(tests))
print("Retrieval sets:", len(retrievals))

# Save copies used in this experiment
tests.to_csv(
    OUT / "test_questions_used.csv",
    index=False,
    encoding="utf-8-sig"
)

with open(
    OUT / "retrievals_used.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        retrievals,
        f,
        ensure_ascii=False,
        indent=2
    )

display(tests.head())

Test questions: 248
Retrieval sets: 248


,id,domain,question,gold
0,passport_001,passport,ই-পাসপোর্ট আবেদনের পাঁচটি মূল ধাপ কী কী?,ই-পাসপোর্ট আবেদনের ৫টি সহজ ধাপ হলো: ১. বর্তমান...
1,passport_179,passport,প্রবাসী শ্রমিকের ৬৪ পৃষ্ঠা ৫ বছর এক্সপ্রেস ই-প...,বিদেশস্থ বাংলাদেশ মিশনে শ্রমিক ও শিক্ষার্থীদের...
2,passport_216,passport,বিদেশে শ্রমিক শিক্ষার্থীর ৬৪ পৃষ্ঠা ৫ বছর এক্স...,বিদেশস্থ বাংলাদেশ মিশনে শ্রমিক ও শিক্ষার্থীদের...
3,passport_241,passport,স্টুডেন্ট হিসেবে মিশনে ৬৪ পৃষ্ঠা ৫ বছর এক্সপ্র...,বিদেশস্থ বাংলাদেশ মিশনে শ্রমিক ও শিক্ষার্থীদের...
4,passport_291,passport,শিশুর পাসপোর্ট সংগ্রহে কী লাগবে?,অপ্রাপ্তবয়স্কের পাসপোর্ট সংগ্রহে পিতা বা মাতা...


In [4]:
# ============================================================
# CELL 3 — LLAMA-3.1-8B RAG INFERENCE
# Stored Top-5 retrievals -> Llama -> predictions.csv
# max_new_tokens = 500
# ============================================================

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from huggingface_hub import notebook_login
from tqdm.auto import tqdm

notebook_login()

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
MAX_NEW_TOKENS = 500

if not torch.cuda.is_available():
    raise RuntimeError("Please enable GPU runtime.")

# 4-bit loading
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

print("Loading Llama-3.1-8B-Instruct...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True
)

model.eval()

device = model.get_input_embeddings().weight.device


# Generate answer from stored contexts
def generate_answer(question, docs):

    context = "\n\n".join([
        f"[Context {i}]\n"
        f"শিরোনাম: {doc.get('title', '')}\n"
        f"তথ্য: {doc.get('text', '')}"
        for i, doc in enumerate(docs, 1)
    ])

    messages = [
        {
            "role": "system",
            "content":
                "তুমি বাংলাদেশের সরকারি সেবা বিষয়ক একজন সহকারী। "
                "শুধু প্রদত্ত Context ব্যবহার করে প্রশ্নের সঠিক ও সংক্ষিপ্ত উত্তর বাংলায় দাও। "
                "Context-এর ভাষা প্রশ্নের থেকে আলাদা হলেও সমার্থক তথ্য বুঝে উত্তর দাও। "
                "ফি, সময়, সংখ্যা এবং প্রয়োজনীয় কাগজপত্র নির্ভুলভাবে উল্লেখ করো। "
                "অপ্রয়োজনীয় ব্যাখ্যা দিও না। "
                "Context-এ উত্তর একেবারেই না থাকলে বলো: "
                "প্রদত্ত তথ্যে এই প্রশ্নের উত্তর পাওয়া যায়নি।"
        },
        {
            "role": "user",
            "content":
                f"Context:\n{context}\n\n"
                f"প্রশ্ন: {question}\n"
                f"উত্তর:"
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=7000
    ).to(device)

    with torch.inference_mode():

        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated = output[0][
        inputs["input_ids"].shape[1]:
    ]

    answer = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    truncated = (
        len(generated) >= MAX_NEW_TOKENS
        and generated[-1].item() != tokenizer.eos_token_id
    )

    return answer, truncated


# Resume support
PARTIAL = OUT / "predictions_partial.csv"

done = {}

if PARTIAL.exists():

    old = pd.read_csv(PARTIAL).fillna("")

    for _, row in old.iterrows():
        done[
            (str(row["domain"]), str(row["id"]))
        ] = row.to_dict()

print("Already completed:", len(done))


# Inference
predictions = []

for i, row in tqdm(
    tests.iterrows(),
    total=len(tests),
    desc="Llama RAG"
):

    key = (
        str(row["domain"]),
        str(row["id"])
    )

    if key in done:
        result = done[key]

    else:

        answer, truncated = generate_answer(
            row["question"],
            retrievals[i]
        )

        result = {
            "id": str(row["id"]),
            "domain": str(row["domain"]),
            "question": row["question"],
            "gold": row["gold"],
            "prediction": answer,
            "truncated": truncated
        }

        done[key] = result

    predictions.append(result)

    # checkpoint
    pd.DataFrame(predictions).to_csv(
        PARTIAL,
        index=False,
        encoding="utf-8-sig"
    )


pred_df = pd.DataFrame(predictions)

pred_df.to_csv(
    OUT / "predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nCompleted:", len(pred_df))

print(
    "Truncated:",
    pred_df["truncated"]
    .astype(str)
    .str.lower()
    .eq("true")
    .sum()
)

print("Saved:", OUT / "predictions.csv")

# Free GPU
del model, tokenizer
gc.collect()
torch.cuda.empty_cache()

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Loading Llama-3.1-8B-Instruct...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Already completed: 0


Llama RAG:   0%|          | 0/248 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Completed: 248
Truncated: 17
Saved: /content/drive/MyDrive/Govt_Chatbot/Llama/RAG/predictions.csv


In [5]:
# ============================================================
# CELL 4 — EVALUATION
# Exact Match, Fuzzy Match, Corpus BLEU,
# ROUGE-1/2/L, Token F1, BERT Precision/Recall/F1
# ============================================================

from collections import Counter
from rapidfuzz import fuzz
from sacrebleu.metrics import BLEU
from bert_score import score as bert_score

df = pd.read_csv(
    OUT / "predictions.csv"
).fillna("")

assert (
    df["gold"]
    .astype(str)
    .str.strip()
    != ""
).all()


# ---------- Normalization ----------
BN_TO_EN = str.maketrans(
    "০১২৩৪৫৬৭৮৯",
    "0123456789"
)

def normalize(text):

    text = unicodedata.normalize(
        "NFKC",
        str(text)
    )

    text = text.translate(BN_TO_EN).lower()

    text = re.sub(
        r"[^\u0980-\u09FFA-Za-z0-9]+",
        " ",
        text
    )

    return re.sub(
        r"\s+",
        " ",
        text
    ).strip()


def tokens(text):
    return normalize(text).split()


# ---------- Exact Match ----------
def exact_match(pred, gold):

    return float(
        normalize(pred) == normalize(gold)
    )


# ---------- Token F1 ----------
def token_f1(pred, gold):

    p = tokens(pred)
    g = tokens(gold)

    if not p or not g:
        return 0.0

    overlap = sum(
        (Counter(p) & Counter(g)).values()
    )

    if overlap == 0:
        return 0.0

    precision = overlap / len(p)
    recall = overlap / len(g)

    return (
        2 * precision * recall
        / (precision + recall)
    )


# ---------- ROUGE-N ----------
def rouge_n(pred, gold, n):

    p = tokens(pred)
    g = tokens(gold)

    if len(p) < n or len(g) < n:
        return 0.0

    pn = Counter(
        tuple(p[i:i+n])
        for i in range(len(p)-n+1)
    )

    gn = Counter(
        tuple(g[i:i+n])
        for i in range(len(g)-n+1)
    )

    overlap = sum((pn & gn).values())

    if overlap == 0:
        return 0.0

    precision = overlap / sum(pn.values())
    recall = overlap / sum(gn.values())

    return (
        2 * precision * recall
        / (precision + recall)
    )


# ---------- ROUGE-L ----------
def rouge_l(pred, gold):

    p = tokens(pred)
    g = tokens(gold)

    if not p or not g:
        return 0.0

    dp = [0] * (len(g) + 1)

    for x in p:

        new = [0]

        for j, y in enumerate(g, 1):

            if x == y:
                new.append(dp[j-1] + 1)

            else:
                new.append(
                    max(dp[j], new[-1])
                )

        dp = new

    lcs = dp[-1]

    precision = lcs / len(p)
    recall = lcs / len(g)

    if precision + recall == 0:
        return 0.0

    return (
        2 * precision * recall
        / (precision + recall)
    )


# ---------- Row metrics ----------
df["Exact Match"] = [
    exact_match(p, g)
    for p, g in zip(df["prediction"], df["gold"])
]

df["Fuzzy Match"] = [
    fuzz.token_set_ratio(
        normalize(p),
        normalize(g)
    ) / 100
    for p, g in zip(df["prediction"], df["gold"])
]

df["Token F1"] = [
    token_f1(p, g)
    for p, g in zip(df["prediction"], df["gold"])
]

df["ROUGE-1"] = [
    rouge_n(p, g, 1)
    for p, g in zip(df["prediction"], df["gold"])
]

df["ROUGE-2"] = [
    rouge_n(p, g, 2)
    for p, g in zip(df["prediction"], df["gold"])
]

df["ROUGE-L"] = [
    rouge_l(p, g)
    for p, g in zip(df["prediction"], df["gold"])
]


# ---------- Corpus BLEU ----------
bleu = BLEU(
    tokenize="none",
    smooth_method="exp",
    effective_order=True
)

pred_bleu = [
    " ".join(tokens(x))
    for x in df["prediction"]
]

gold_bleu = [
    " ".join(tokens(x))
    for x in df["gold"]
]

corpus_bleu = (
    bleu.corpus_score(
        pred_bleu,
        [gold_bleu]
    ).score
    / 100
)


# ---------- BERTScore ----------
print("Calculating BERTScore...")

P, R, F1 = bert_score(
    df["prediction"].astype(str).tolist(),
    df["gold"].astype(str).tolist(),
    model_type="bert-base-multilingual-cased",
    batch_size=4,
    device="cpu",
    verbose=True,
    idf=False,
    rescale_with_baseline=False
)

df["BERT Precision"] = P.cpu().numpy()
df["BERT Recall"] = R.cpu().numpy()
df["BERT F1"] = F1.cpu().numpy()


# ---------- Final result ----------
result = pd.DataFrame({

    "metric": [
        "Exact Match",
        "Fuzzy Match",
        "Corpus BLEU",
        "ROUGE-1",
        "ROUGE-2",
        "ROUGE-L",
        "Token F1",
        "BERT Precision",
        "BERT Recall",
        "BERT F1"
    ],

    "score": [
        df["Exact Match"].mean(),
        df["Fuzzy Match"].mean(),
        corpus_bleu,
        df["ROUGE-1"].mean(),
        df["ROUGE-2"].mean(),
        df["ROUGE-L"].mean(),
        df["Token F1"].mean(),
        df["BERT Precision"].mean(),
        df["BERT Recall"].mean(),
        df["BERT F1"].mean()
    ]
})


df.to_csv(
    OUT / "predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

result.to_csv(
    OUT / "result.csv",
    index=False,
    encoding="utf-8-sig"
)

display(result)

print("\nSaved:")
print(OUT / "test_questions_used.csv")
print(OUT / "retrievals_used.json")
print(OUT / "predictions_partial.csv")
print(OUT / "predictions.csv")
print(OUT / "result.csv")

Calculating BERTScore...


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

computing greedy matching.


  0%|          | 0/62 [00:00<?, ?it/s]

done in 37.75 seconds, 6.57 sentences/sec


,metric,score
0,Exact Match,0.060484
1,Fuzzy Match,0.730459
2,Corpus BLEU,0.338009
3,ROUGE-1,0.449158
4,ROUGE-2,0.370512
5,ROUGE-L,0.433204
6,Token F1,0.449158
7,BERT Precision,0.799819
8,BERT Recall,0.796920
9,BERT F1,0.796606



Saved:
/content/drive/MyDrive/Govt_Chatbot/Llama/RAG/test_questions_used.csv
/content/drive/MyDrive/Govt_Chatbot/Llama/RAG/retrievals_used.json
/content/drive/MyDrive/Govt_Chatbot/Llama/RAG/predictions_partial.csv
/content/drive/MyDrive/Govt_Chatbot/Llama/RAG/predictions.csv
/content/drive/MyDrive/Govt_Chatbot/Llama/RAG/result.csv
